In [ ]:
from huggingface_hub import login
login()

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


### Préparation de l'environnement
Assurez-vous d'avoir installé les bibliothèques requises : `pip install -U transformers datasets peft bitsandbytes accelerate` et d'être connecté à Hugging Face via `huggingface-cli login` si nécessaire.

In [ ]:
!pip install -U sentencepiece tiktoken bitsandbytes>=0.46.1 accelerate peft datasets transformers

In [ ]:
import torch
import sentencepiece
import tiktoken
from transformers import AutoTokenizer, LlamaTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# ==========================================
# ÉTAPE 1 OBLIGATOIRE - Fusion des tokenizers
# ==========================================

model_id = "meta-llama/Llama-3.2-1B"
baoule_tokenizer_id = "Adjoumani/baoule_tokenizer"

# 1. & 2. Charger les tokenizers
print("Chargement du tokenizer Llama...")
llama_tokenizer = AutoTokenizer.from_pretrained(model_id)

print("Chargement du tokenizer Baoulé...")
# On force l'utilisation de LlamaTokenizer (version lente/classique) pour éviter l'erreur de conversion
try:
    baoule_tokenizer = LlamaTokenizer.from_pretrained(baoule_tokenizer_id, use_fast=False)
except Exception as e:
    print(f"Avertissement LlamaTokenizer classique : {e}")
    print("Essai avec AutoTokenizer + trust_remote_code...")
    baoule_tokenizer = AutoTokenizer.from_pretrained(baoule_tokenizer_id, trust_remote_code=True, use_fast=False)

# Llama 3.x nécessite un pad_token, on utilise eos_token
if llama_tokenizer.pad_token is None:
    llama_tokenizer.pad_token = llama_tokenizer.eos_token

# 3. Identifier les nouveaux tokens baoulé
llama_vocab = set(llama_tokenizer.get_vocab().keys())
baoule_vocab = set(baoule_tokenizer.get_vocab().keys())
new_tokens = list(baoule_vocab - llama_vocab)
print(f"Nombre de tokens baoulé absents du vocabulaire Llama : {len(new_tokens)}")

# 4. Ajouter les tokens au tokenizer Llama
llama_tokenizer.add_tokens(new_tokens)

# Configuration QLoRA pour le modèle (NF4)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

# Charger le modèle
print("Chargement du modèle Llama-3.2-1B...")
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto"
)

# 5. Redimensionner les embeddings du modèle
model.resize_token_embeddings(len(llama_tokenizer))

# 6. Sauvegarder le tokenizer fusionné
merged_tokenizer_path = "./merged_llama_baoule_tokenizer"
llama_tokenizer.save_pretrained(merged_tokenizer_path)
print(f"Tokenizer fusionné sauvegardé dans : {merged_tokenizer_path}")

Chargement du tokenizer Llama...
Chargement du tokenizer Baoulé...
Nombre de tokens baoulé absents du vocabulaire Llama : 3
Chargement du modèle Llama-3.2-1B...


Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

[transformers] The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


Tokenizer fusionné sauvegardé dans : ./merged_llama_baoule_tokenizer


### Étape 2 : Tokenisation et calcul dynamique des époques (Beyond Chinchilla's Ω)

In [ ]:
from datasets import load_dataset
import math

# ===================================================================
# ÉTAPE 2 OBLIGATOIRE - Tokenisation des corpus et recomptage réel
# ===================================================================

# Chemins vers les fichiers (ajustés selon votre environnement)
file_baoule = '/content/corpus_baoule_pretrain_clean (1).txt'
file_francais = '/content/corpus_francais_145M.txt'

# Chargement brut
ds_baoule = load_dataset('text', data_files=file_baoule, split='train')
ds_francais = load_dataset('text', data_files=file_francais, split='train')

# 1. Tokenisation (utilisation EXCLUSIVE du tokenizer fusionné)
def tokenize_function(examples):
    return llama_tokenizer(examples["text"], add_special_tokens=False)

print("Tokenisation du corpus baoulé...")
tokenized_baoule = ds_baoule.map(tokenize_function, batched=True, remove_columns=["text"])
print("Tokenisation du corpus français...")
tokenized_francais = ds_francais.map(tokenize_function, batched=True, remove_columns=["text"])

# 2. Comptage réel des tokens
def count_tokens(dataset):
    return sum([len(item) for item in dataset['input_ids']])

real_tokens_baoule = count_tokens(tokenized_baoule)
real_tokens_francais = count_tokens(tokenized_francais)
total_real_tokens = real_tokens_baoule + real_tokens_francais

# 3. Affichage des comptes réels
print(f"\n--- COMPTAGES RÉELS (Post-Tokenisation) ---")
print(f"Tokens Baoulé   : {real_tokens_baoule:,}")
print(f"Tokens Français : {real_tokens_francais:,}")
print(f"Total Tokens    : {total_real_tokens:,}")
print(f"Ratio obtenu    : {real_tokens_baoule/total_real_tokens:.2%} Baoulé / {real_tokens_francais/total_real_tokens:.2%} Français")

# 4. Calcul dynamique des époques (Cadre Beyond Chinchilla, Contexte B)
# N = 1.23 * 10^9 paramètres.
N_params = 1.23e9
# Demp_total = 2 * N = 2.46 * 10^9 tokens
demp_total = 2 * N_params
# Split linguistique appliqué à Demp : 70% Baoulé
demp_baoule = 0.70 * demp_total

# Époques = Demp_baoulé / D_réel_baoulé_tokenisé (arrondi supérieur)
calculated_epochs = math.ceil(demp_baoule / real_tokens_baoule)

print(f"\n--- CALCUL BEYOND CHINCHILLA (Contexte B) ---")
print(f"Demp Total visé  : {demp_total:,.0f} tokens")
print(f"Demp Baoulé (70%): {demp_baoule:,.0f} tokens")
print(f"Époques Phase B à appliquer : {calculated_epochs} (Demp_baoulé / tokens réels baoulé)")

Tokenisation du corpus baoulé...


Map:   0%|          | 0/1374237 [00:00<?, ? examples/s]

Tokenisation du corpus français...


Map:   0%|          | 0/6091041 [00:00<?, ? examples/s]


--- COMPTAGES RÉELS (Post-Tokenisation) ---
Tokens Baoulé   : 319,833,132
Tokens Français : 177,869,611
Total Tokens    : 497,702,743
Ratio obtenu    : 64.26% Baoulé / 35.74% Français

--- CALCUL BEYOND CHINCHILLA (Contexte B) ---
Demp Total visé  : 2,460,000,000 tokens
Demp Baoulé (70%): 1,722,000,000 tokens
Époques Phase B à appliquer : 6 (Demp_baoulé / tokens réels baoulé)


### Étape 3 : Packing et Entrelacement (Interleaving)

In [ ]:
from datasets import interleave_datasets

# =====================================================================
# ÉTAPE 3 OBLIGATOIRE - Mélange interleaved des deux langues
# =====================================================================

BLOCK_SIZE = 2048

# 1. Fonction de packing
def group_texts(examples):
    concatenated_examples = {k: sum(examples[k], []) for k in examples.keys()}
    total_length = len(concatenated_examples[list(examples.keys())[0]])
    total_length = (total_length // BLOCK_SIZE) * BLOCK_SIZE
    result = {
        k: [t[i : i + BLOCK_SIZE] for i in range(0, total_length, BLOCK_SIZE)]
        for k, t in concatenated_examples.items()
    }
    # Ajouter les labels pour l'entraînement causal LM
    result["labels"] = result["input_ids"].copy()
    return result

print("Packing du corpus baoulé...")
packed_baoule = tokenized_baoule.map(group_texts, batched=True)
print("Packing du corpus français...")
packed_francais = tokenized_francais.map(group_texts, batched=True)

# 2. Fusion interleaved
# 3. EXPLICATION : L'entraînement séquentiel (tout le baoulé puis tout le français) est
# absolument à proscrire car il provoque un "oubli catastrophique" (catastrophic forgetting).
# Le modèle oublierait les spécificités de la première langue en apprenant la seconde.
# L'interleaving garantit une exposition simultanée et proportionnelle (70/30) à chaque pas de gradient.
print("Mélange (interleave) des datasets à 70% Baoulé / 30% Français...")
mixed_dataset = interleave_datasets(
    [packed_baoule, packed_francais],
    probabilities=[0.70, 0.30],
    stopping_strategy="all_exhausted"
)

# 4. Shuffle final
mixed_dataset = mixed_dataset.shuffle(seed=42)
print("Dataset final prêt.")

Packing du corpus baoulé...


Map:   0%|          | 0/1374237 [00:00<?, ? examples/s]

Packing du corpus français...


Map:   0%|          | 0/6091041 [00:00<?, ? examples/s]

Mélange (interleave) des datasets à 70% Baoulé / 30% Français...
Dataset final prêt.


### Étape 4 : Configuration LoRA et Lancement de l'Entraînement

In [ ]:
!pip install -U "torchao>=0.16.0"

import torch
import gc
import os
import numpy as np
from transformers import AutoModelForCausalLM, TrainingArguments, Trainer
from transformers.trainer_utils import get_last_checkpoint
from peft import LoraConfig, get_peft_model

# =====================================================================
# OPTIMISATION A100 : BFloat16 + SDPA + Fused Optimizer + TOKEN ACCURACY
# =====================================================================

# Nettoyage profond du cache CUDA
try:
    del trainer
except NameError:
    pass
try:
    del model
except NameError:
    pass
gc.collect()
torch.cuda.empty_cache()

print("Rechargement du modèle avec accélération SDPA...")
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    attn_implementation="sdpa" # <-- Accélération de l'attention native PyTorch
)

# Redimensionner les embeddings
model.resize_token_embeddings(len(llama_tokenizer))

# RÉACTIVÉ : Indispensable pour éviter le crash mémoire (OOM) avec un vocabulaire étendu
model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})

lora_config = LoraConfig(
    r=64,
    lora_alpha=128,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    modules_to_save=["embed_tokens", "lm_head"],
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

output_dir = "/content/drive/MyDrive/Llama3.2-1B-Baoule-Fr-LoRA-Rapide"

# --- AJOUT POUR LA TOKEN ACCURACY ---
print("Préparation du dataset d'évaluation (0.5% du dataset global)...")
split_dataset = mixed_dataset.train_test_split(test_size=0.005, seed=42)
train_data = split_dataset['train']
eval_data = split_dataset['test']

def preprocess_logits_for_metrics(logits, labels):
    if isinstance(logits, tuple):
        logits = logits[0]
    return logits.argmax(dim=-1)

def compute_metrics(eval_preds):
    preds, labels = eval_preds
    # labels sont paddés avec -100, on les ignore dans le calcul
    mask = labels != -100
    correct = (preds[mask] == labels[mask])
    accuracy = correct.sum() / mask.sum()
    return {"token_accuracy": accuracy}
# ------------------------------------

# Arguments d'entraînement ajustés pour la VITESSE MAXIMALE SANS OOM
training_args = TrainingArguments(
    output_dir=output_dir,
    num_train_epochs=calculated_epochs,
    per_device_train_batch_size=4,      # Batch de 4 (maximum stable avec Checkpointing)
    gradient_accumulation_steps=8,      # <-- RÉDUIT À 8 pour des mises à jour 4x plus rapides !
    learning_rate=2e-4,
    logging_steps=10,
    eval_strategy="steps",              # <-- ÉVALUATION ACTIVÉE
    eval_steps=100,                     # <-- Calcule la token accuracy tous les 100 pas
    save_strategy="steps",
    save_steps=500,                     # SAUVEGARDE AUTO TOUS LES 500 PAS
    optim="adamw_torch_fused",          # <-- OPTIMISEUR FUSIONNÉ : Beaucoup plus rapide sur A100
    bf16=True,
    max_grad_norm=0.3,
    warmup_steps=100,
    lr_scheduler_type="cosine",
    report_to="none",
    dataloader_num_workers=4,
    dataloader_pin_memory=True          # <-- Accélère le transfert RAM -> VRAM
)

# Initialisation du Trainer
trainer = Trainer(
    model=model,
    train_dataset=train_data,           # Dataset d'entraînement réduit
    eval_dataset=eval_data,             # Dataset d'évaluation
    compute_metrics=compute_metrics,    # Fonction de précision ajoutée
    preprocess_logits_for_metrics=preprocess_logits_for_metrics, # Prévient les fuites RAM (OOM)
    args=training_args,
)

# ---------------------------------------------------------
# GESTION INTELLIGENTE DES REPRISES (CHUNKS)
# ---------------------------------------------------------
last_checkpoint = get_last_checkpoint(output_dir)
if last_checkpoint is not None:
    print(f"\n>>> REPRISE DE L'ENTRAÎNEMENT DÉTECTÉE DEPUIS : {last_checkpoint} <<<")
else:
    print("\n>>> DÉBUT DE L'ENTRAÎNEMENT À ZÉRO <<<")


trainer.train(resume_from_checkpoint=last_checkpoint)


Rechargement du modèle avec accélération SDPA...


Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/peft/tuners/tuners_utils.py:1377: UserWarning: Model has `tie_word_embeddings=True` and a tied layer is part of the adapter, but `ensure_weight_tying` is not set to True. This can lead to complications, for example when merging the adapter or converting your model to formats other than safetensors. Check the discussion here: https://github.com/huggingface/peft/issues/2777
  warnings.warn(msg)


trainable params: 570,437,632 || all params: 1,806,258,176 || trainable%: 31.5812
Préparation du dataset d'évaluation (0.5% du dataset global)...

>>> REPRISE DE L'ENTRAÎNEMENT DÉTECTÉE DEPUIS : /content/drive/MyDrive/Llama3.2-1B-Baoule-Fr-LoRA-Rapide/checkpoint-500 <<<


[transformers] Warning: The following arguments do not match the ones in the `trainer_state.json` within the checkpoint directory: 
	eval_steps: 100 (from args) != 500 (from trainer_state.json)


Step,Training Loss,Validation Loss,Token Accuracy
1000,1.332161,1.350688,0.000970


/usr/local/lib/python3.13/dist-packages/peft/utils/save_and_load.py:452: UserWarning: Setting `save_embedding_layers` to `True` as the embedding layer has been resized during finetuning.
  warnings.warn(


KeyboardInterrupt: 

### TEST RAPIDE D'UN CHECKPOINT (Inférence à la volée)
Utilisez cette cellule pour tester la génération en baoulé d'un checkpoint spécifique sans avoir à fusionner le modèle complet.

In [ ]:
# Test instantané sans recharger le modèle
prompt = "Baoulé : nglɛmun kpa, a ti sɛ?\nFrançais :"

print("================ GÉNÉRATION EN COURS =================")
inputs = tokenizer(nouveau_prompt, return_tensors="pt").to("cuda")

with torch.no_grad():
    outputs = model_to_test.generate(
        **inputs,
        max_new_tokens=40,
        temperature=0.3,
        top_p=0.9,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id
    )

resultat = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(f"\n[TEST] : \n{resultat}")
print("-"*50)


================ GÉNÉRATION EN COURS =================

[TEST] : 
Baoulé : nglɛmun kpa, a ti sɛ?
Français : baoulé : nglɛmun kpa, a ti sɛ? ?Amun a bu i le kɛ amun su kɔ amun ɲrun
--------------------------------------------------


In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

adapter_path = "/content/drive/MyDrive/Llama3.2-1B-Baoule-Fr-LoRA-Rapide/checkpoint-1000"
base_model_id = "meta-llama/Llama-3.2-1B"
tokenizer_path = "./merged_llama_baoule_tokenizer"

# On vérifie si base_model est toujours en mémoire, sinon on le recharge
try:
    base_model
except NameError:
    print("Le modèle de base n'est plus en mémoire. Rechargement en cours...")
    tokenizer = AutoTokenizer.from_pretrained(tokenizer_path)
    base_model = AutoModelForCausalLM.from_pretrained(
        base_model_id,
        torch_dtype=torch.bfloat16,
        device_map="auto"
    )
    base_model.resize_token_embeddings(len(tokenizer))

print(f"\nChargement de l'adaptateur depuis : {adapter_path}...")
# 1. Chargement de l'adaptateur PEFT par-dessus le modèle de base
peft_model = PeftModel.from_pretrained(base_model, adapter_path)

# 2 & 3. Vérification des modules et ratio de paramètres
print("\n--- VÉRIFICATION DES PARAMÈTRES ENTRAÎNABLES ---")
peft_model.print_trainable_parameters()

print("\nFusion des poids LoRA dans le modèle de base...")
merged_model = peft_model.merge_and_unload()

print("Fusion mathématique terminée avec succès.")


Le modèle de base n'est plus en mémoire. Rechargement en cours...


Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]


Chargement de l'adaptateur depuis : /content/drive/MyDrive/Llama3.2-1B-Baoule-Fr-LoRA-Rapide/checkpoint-1000...


/usr/local/lib/python3.13/dist-packages/peft/tuners/tuners_utils.py:1377: UserWarning: Model has `tie_word_embeddings=True` and a tied layer is part of the adapter, but `ensure_weight_tying` is not set to True. This can lead to complications, for example when merging the adapter or converting your model to formats other than safetensors. Check the discussion here: https://github.com/huggingface/peft/issues/2777
  warnings.warn(msg)



--- VÉRIFICATION DES PARAMÈTRES ENTRAÎNABLES ---
trainable params: 0 || all params: 1,806,258,176 || trainable%: 0.0000

Fusion des poids LoRA dans le modèle de base...
Fusion mathématique terminée avec succès.


/usr/local/lib/python3.13/dist-packages/peft/tuners/tuners_utils.py:687: UserWarning: Input and output embeddings are no longer tied after merging. Setting `tie_word_embeddings=False` in the model config.
  warnings.warn(


In [ ]:
import os
import gc
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

output_merged_dir = "/content/drive/MyDrive/Llama3.2-1B-Baoule-Fr-Final-Merged"
os.makedirs(output_merged_dir, exist_ok=True)

print(f"Sauvegarde du modèle et du tokenizer en safetensors dans {output_merged_dir}...")
merged_model.save_pretrained(output_merged_dir, safe_serialization=True)
tokenizer.save_pretrained(output_merged_dir)
print("Sauvegarde terminée avec succès !")

print("\nNettoyage mémoire avant le test de génération...")
try:
    del merged_model
except NameError:
    pass
try:
    del peft_model
except NameError:
    pass
try:
    del base_model
except NameError:
    pass
gc.collect()
torch.cuda.empty_cache()

print("\nRechargement du modèle final fusionné DEPUIS LE DISQUE...")
# 1. On recharge depuis le dossier de sauvegarde pour s'assurer que les fichiers sont valides
final_model = AutoModelForCausalLM.from_pretrained(
    output_merged_dir,
    torch_dtype=torch.bfloat16,
    device_map="auto"
)
final_tokenizer = AutoTokenizer.from_pretrained(output_merged_dir)

# 2. Test de génération sur votre prompt Baoulé
prompt_test = "Ngwlɛlɛ nga sran’m be fa yo ninnge mun’n, yɛle"
print(f"\nPrompt envoyé : '{prompt_test}'")

inputs = final_tokenizer(prompt_test, return_tensors="pt").to("cuda")

outputs = final_model.generate(
    **inputs,
    max_new_tokens=40,
    temperature=0.3,
    top_p=0.9,
    do_sample=True,
    pad_token_id=final_tokenizer.eos_token_id
)

texte_genere = final_tokenizer.decode(outputs[0], skip_special_tokens=True)
print("\n--- RÉSULTAT DU TEST --- ")
print(texte_genere)
print("\nSi le texte a du sens et utilise les tokens Baoulé, la fusion est un succès complet !")


Sauvegarde du modèle et du tokenizer en safetensors dans /content/drive/MyDrive/Llama3.2-1B-Baoule-Fr-Final-Merged...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Sauvegarde terminée avec succès !

Nettoyage mémoire avant le test de génération...

Rechargement du modèle final fusionné DEPUIS LE DISQUE...


Loading weights:   0%|          | 0/147 [00:00<?, ?it/s]


Prompt envoyé : 'Ngwlɛlɛ nga sran’m be fa yo ninnge mun’n, yɛle'

--- RÉSULTAT DU TEST --- 
Ngwlɛlɛ nga sran’m be fa yo ninnge mun’n, yɛle kpanngban kpa. I sɔ’n ti’n, kɛ be fa ninnge nga be fa yo ninnge mun’n be di junman’n, �

Si le texte a du sens et utilise les tokens Baoulé, la fusion est un succès complet !
